# EduVision_DV: Data Cleaning & Transformation Pipeline
### Higher Education Performance Analytics (QS & Times Higher Education Rankings)
**Author:** EduVision Analytics Team  
**Dataset:** Multi-Source University Ranking Panel (2020–2024)  
**Deliverable:** Module 2 - `education_cleaning.ipynb` & `data/university_cleaned.csv`  

---
### Pipeline Overview:
1. **Data Ingestion & Integrity Check**: Load raw dataset, verify dimensions, inspect initial schema.
2. **Deduplication**: Identify and eliminate duplicate institutional records across academic years.
3. **Entity Standardization**: Standardize institution names, country names, and map ISO-3 country codes.
4. **Missing Value Imputation**: Analyze null distribution and apply regional-year median imputation.
5. **Indicator Normalization**: Bound performance metrics (0-100 scale) and validate data types.
6. **Tableau-Ready Export**: Save the standardized dataset to `data/university_cleaned.csv`.


In [ ]:
import pandas as pd
import numpy as np
import os

raw_path = os.path.join("..", "data", "university_raw_data.csv")
if not os.path.exists(raw_path):
    raw_path = os.path.join("data", "university_raw_data.csv")

print(f"Loading raw data from: {raw_path}")
df_raw = pd.read_csv(raw_path)
print(f"Initial Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()


### 2. Deduplication & Identifier Verification
Ensure no institution is listed multiple times within the same academic year.


In [ ]:
initial_len = len(df_raw)
df_dedup = df_raw.drop_duplicates(subset=["university_name", "year"]).copy()
dupes_removed = initial_len - len(df_dedup)
print(f"Duplicates removed: {dupes_removed}")
print(f"Unique institutions in 2024: {df_dedup[df_dedup['year'] == 2024]['university_name'].nunique():,}")


### 3. Entity Standardization (Institutions & Countries)
Standardize university naming formats (including parenthetical acronyms for globally recognized institutions) and map country names to standard 3-letter ISO-3166 alpha-3 codes for seamless Tableau geocoding.


In [ ]:
# 1. Standardize University Names
name_replacements = {
    "Massachusetts Institute of Technology": "MIT (Massachusetts Institute of Technology)",
    "UCL": "UCL (University College London)",
    "ETH Zurich": "ETH Zurich (Swiss Federal Institute of Technology)",
    "National University of Singapore": "National University of Singapore (NUS)",
    "California Institute of Technology": "Caltech (California Institute of Technology)"
}
df_dedup["university_name_standardized"] = df_dedup["university_name"].replace(name_replacements).str.strip()

# 2. Standardize Country Names
country_standardization = {
    "United States": "USA",
    "United Kingdom": "UK",
    "South Korea": "South Korea",
    "Hong Kong": "Hong Kong (SAR)",
    "Russian Federation": "Russia"
}
df_dedup["country_clean"] = df_dedup["country"].replace(country_standardization)

# 3. ISO Country Codes
iso_map = {
    "USA": "USA", "UK": "GBR", "Switzerland": "CHE", "Singapore": "SGP",
    "Canada": "CAN", "Australia": "AUS", "China": "CHN", "Japan": "JPN",
    "Germany": "DEU", "France": "FRA", "South Korea": "KOR", "Hong Kong (SAR)": "HKG",
    "Brazil": "BRA", "Chile": "CHL", "Netherlands": "NLD", "Sweden": "SWE",
    "Italy": "ITA", "Spain": "ESP", "Belgium": "BEL", "Denmark": "DNK",
    "Norway": "NOR", "Finland": "FIN", "Austria": "AUT", "Ireland": "IRL",
    "Poland": "POL", "Czech Republic": "CZE", "Mexico": "MEX", "India": "IND",
    "Taiwan": "TWN", "Malaysia": "MYS", "Saudi Arabia": "SAU", "United Arab Emirates": "ARE",
    "Israel": "ISR", "Turkey": "TUR", "Thailand": "THA", "Indonesia": "IDN",
    "New Zealand": "NZL", "Fiji": "FJI", "Argentina": "ARG", "Colombia": "COL",
    "Peru": "PER", "Uruguay": "URY", "South Africa": "ZAF"
}
df_dedup["country_iso_code"] = df_dedup["country_clean"].map(iso_map).fillna("OTH")
print("Top 10 Countries by University Representation:")
print(df_dedup[df_dedup['year'] == 2024]['country_clean'].value_counts().head(10))


### 4. Missing Value Analysis & Regional Imputation
Target: Missing values strictly below **2%** across all features.


In [ ]:
null_counts = df_dedup.isnull().sum()
print("Missing values per column before imputation:")
print(null_counts[null_counts > 0])

# Impute sustainability score using regional median within each academic year
df_dedup["sustainability_score"] = df_dedup.groupby(["region", "year"])["sustainability_score"].transform(
    lambda x: x.fillna(x.median())
)
df_dedup["sustainability_score"] = df_dedup["sustainability_score"].fillna(df_dedup["sustainability_score"].median())

total_missing = df_dedup.isnull().sum().sum()
missing_rate = (total_missing / (df_dedup.shape[0] * df_dedup.shape[1])) * 100.0
print(f"\nRemaining Missing Values: {total_missing} ({missing_rate:.4f}%)")
assert missing_rate < 2.0, "Missing rate exceeds 2% threshold!"
print("[PASS] Evaluation Criteria Met: Less than 2% missing values.")


### 5. Metric Normalization & Indicator Bounds
Ensure all indicator scores are strictly bounded within $[0, 100]$ and ratios are physically valid.


In [ ]:
score_cols = [
    "raw_overall_score", "academic_reputation_score", "employer_reputation_score",
    "faculty_student_score", "citations_per_faculty", "intl_faculty_ratio",
    "sustainability_score", "employment_outcomes_score", "the_teaching_score",
    "the_research_environment", "the_research_quality", "the_industry_score",
    "the_international_outlook"
]

for col in score_cols:
    df_dedup[col] = df_dedup[col].clip(0.0, 100.0).round(1)

df_dedup["student_faculty_ratio"] = df_dedup["student_faculty_ratio"].clip(1.0, 60.0).round(1)
df_dedup["international_student_pct"] = df_dedup["international_student_pct"].clip(0.0, 100.0).round(1)

# Re-compute yearly ranks cleanly
df_dedup["cleaned_rank"] = df_dedup.groupby("year")["raw_overall_score"].rank(ascending=False, method="min").astype(int)

df_dedup.describe().round(2)


### 6. Export Tableau-Ready Cleaned Dataset
Save the cleaned dataset to `data/university_cleaned.csv` ready for KPI engineering and Tableau dashboard ingestion.


In [ ]:
out_path = os.path.join("..", "data", "university_cleaned.csv")
if not os.path.exists(os.path.dirname(out_path)):
    out_path = os.path.join("data", "university_cleaned.csv")

df_dedup.to_csv(out_path, index=False)
print(f"[SUCCESS] Cleaned dataset written to: {out_path}")
print(f"Final Records: {len(df_dedup):,} rows | {len(df_dedup.columns)} columns")
